### Day 10 · OOP 多态+契约 (L3)

今天进入 OOP 最强大的武器:**多OP 最强大的武器:**多态**与**契约**。
你会学会:用同一接口处理不同对象、用 abc 强制子类履约。

> 叙事锚点:电商 v2 —— Payment(abc) 支付系统
> 教学法:**NCDL**(负案例驱动)完整落地 —— 先踩坑,再学


> **Step 0 · 痛点** —— 120 行 if-elif 的灾难

假设你要写一个支付函数,支持支付宝/微信/银行卡:


In [ ]:
def pay(payment_type, amount):
    if payment_type == "alipay":
        # 30 行支付宝逻辑
        print(f"支付宝支付 {amount} 元")
    elif payment_type == "wechat":
        # 30 行微信逻辑
        print(f"微信支付 {amount} 元")
    elif payment_type == "bank":
        # 30 行银行卡逻辑
        print(f"银行卡支付 {amount} 元")
    else:
        raise ValueError("未知支付类型")

pay("alipay", 99.0)
pay("wechat", 50.0)


> **逐行解剖**  
① `if-elif` 链:每新增一种支付都要**改这个函数**  
② 函数越来越长,阅读/维护成本指数上升  
③ 测试要覆盖所有分支,新增一种就要加测试  
④ **核心问题**:类型和逻辑耦合在一起


> **3 个致命问题**  
1. **新增 = 改核心**:加一种支付方式必须动 `pay` 函数  
2. **函数膨胀**:120 行 if-else,改一行可能引入 3 个 bug  
3. **测试爆炸**:N 种支付 × M 种场景 = N×M 个用例

**今天的目标**:`pay` 函数缩小到 4 行,新增支付方式只需**加文件**。


> **重构思路** —— 把每种支付抽取成类

**核心洞察**:每种支付方式都有"执行支付"这个共同行为。
与其用 if-elif 判断类型,不如让每种支付**自己知道怎么支付**。

**重构三步**:  
1. 把每种支付的逻辑封装成类  
2. 每个类都实现 `execute(amount)` 方法  
3. 调用方只调用 `execute`,不判断类型


In [ ]:
class Alipay:
    def execute(self, amount):
        return f"支付宝支付 {amount} 元"

class WeChatPay:
    def execute(self, amount):
        return f"微信支付 {amount} 元"

class BankPay:
    def execute(self, amount):
        return f"银行卡支付 {amount} 元"

# --- 执行过程 ---
# 第 1-3 行:定义 Alipay 类,注册 execute 方法(暂不执行)
# 第 5-7 行:定义 WeChatPay 类,注册 execute 方法
# 第 9-11 行:定义 BankPay 类,注册 execute 方法
# 三个类互不继承,但都有 execute → 这就是多态的基础

alipay = Alipay()
wechat = WeChatPay()
bank = BankPay()
print(alipay.execute(99.0))
print(wechat.execute(50.0))
print(bank.execute(200.0))


> **逐行解剖**  
① 三个类**没有继承同一个父类**  
② 但它们都有 `execute` 方法 → 这就是**鸭子类型**  
③ 调用时直接 `obj.execute()`,不需要 if-elif  
④ 新增一种支付?再加一个类即可,**原有代码零修改**


> **ASCII 内存图** —— 三个独立类

```
Alipay 类 ──► execute(self, amount)
WeChatPay 类 ──► execute(self, amount)
BankPay 类 ──► execute(self, amount)

alipay = Alipay()    →  {execute: <function>}
wechat = WeChatPay()  →  {execute: <function>}
bank   = BankPay()    →  {execute: <function>}

三个对象互不相关,但都有 execute 方法
```


#### 鸭子类型 (Duck Typing) 多态

> **类比**  
'像鸭子就是鸭子':走路像鸭子、叫声像鸭子,它就是鸭子,
不管它**继承自谁**。Python 只看'有没有这个方法'。

> **解释**  
Python 不关注对象是哪个类的实例,
只关心'**有没有这个方法**'。
只要对象有 `execute` 方法,就能传给 `checkout`。

> **为什么**叫鸭子类型?  
源自谚语:'如果它走起来像鸭子,叫起来像鸭子,
那它就是鸭子'。


In [ ]:
class Alipay:
    def execute(self, amount):
        return f"支付宝支付 {amount} 元"

class WeChatPay:
    def execute(self, amount):
        return f"微信支付 {amount} 元"

def checkout(cart_total, payment):
    # payment 可以是任何有 execute 方法的对象
    return payment.execute(cart_total)

# --- 执行过程 ---
# 第 10 行 checkout(99.0, Alipay()):
#   ① 创建 Alipay 实例
#   ② 调用 checkout → payment 指向该实例
#   ③ payment.execute(99.0) → 找到 Alipay.execute
#   ④ 返回 "支付宝支付 99.0 元"
# 第 11 行 checkout(50.0, WeChatPay()):同理

print(checkout(99.0, Alipay()))
print(checkout(50.0, WeChatPay()))


> **逐行解剖**  
① `checkout` 函数**不判断类型**,只调用 `payment.execute`  
② 传入 `Alipay()` → 执行支付宝逻辑  
③ 传入 `WeChatPay()` → 执行微信逻辑  
④ **完全去掉 if-elif**!新增支付方式无需修改 `checkout`


In [ ]:
# 甚至可以是'临时'的类 —— 鸭子类型的灵活性
class CreditCard:
    def execute(self, amount):
        return f"信用卡支付 {amount} 元"

class Bitcoin:
    def execute(self, amount):
        return f"比特币支付 {amount} 元"

def checkout(cart_total, payment):
    result = payment.execute(cart_total)
    print(f"支付结果: {result}")
    return result

# --- 执行过程 ---
# ① CreditCard 和 Bitcoin 都是'临时'写的类
# ② 不继承任何父类,但都有 execute 方法
# ③ checkout 来者不拒,照单全收
# ④ 这就是鸭子类型的极致灵活

for p in [CreditCard(), Bitcoin()]:
    checkout(100.0, p)


> **逐行解剖**  
① `CreditCard` 和 `Bitcoin` 没有共同父类  
② 但都实现了 `execute` 方法 → 鸭子类型认可它们  
③ `for p in [...]` → 统一处理不同类型的支付  
④ 灵活性极高,但也**失去了保护**


> **ASCII 内存图** —— 鸭子类型调用链

```
checkout(99.0, Alipay())
    │
    ▼
payment = Alipay 实例
    │
    ▼
payment.execute(99.0)
    │
    ▼
Alipay.execute(self=实例, amount=99.0)
    │
    ▼
返回 "支付宝支付 99.0 元"

关键:checkout 不关心 payment 是哪个类
只要它有 execute 方法就行 → 鸭子类型


> **为什么**鸭子类型强大?  
- **扩展性**:新增一种支付只需加一个类  
- **解耦**:调用方和被调用方互不依赖  
- **简洁**:消灭 if-elif,代码量骤降


> **NCDL Break It #1** —— 鸭子类型的代价

鸭子类型的致命弱点:**漏写方法不会提前报错**。
代码能创建对象,能调用函数,直到**运行到那一行**才爆炸。


In [ ]:
# ============ BREAK IT 演示 ============
class BrokenAlipay:
    # 注意:execute 方法被漏写了!
    pass

def checkout(total, payment):
    return payment.execute(total)

alipay = BrokenAlipay()
print("创建成功,一切看似正常...")
try:
    checkout(99.0, alipay)
except AttributeError as e:
    print(f"报错: {e}")
    print("错误在运行中才暴露,难以追溯!")
# ============ END BREAK IT ============


> **逐行解剖**  
① `BrokenAlipay` 没有 `execute` 方法 → 但**创建不报错**  
② `alipay = BrokenAlipay()` 成功 → 看起来一切正常  
③ `checkout(99.0, alipay)` → 调用 `payment.execute`  
④ `AttributeError` → 此时才暴露问题  
⑤ **问题**:bug 在**运行时**才出现,可能藏很久


> **常见错误**  
1. **拼写错误**:`execte` vs `execute` → 不报错,运行才炸  
2. **参数不一致**:`execute(self, amount, currency)` → 调用时参数不匹配  
3. **忘记实现**:子类漏写方法 → 创建成功,调用才失败

**结论**:个人脚本用鸭子类型 OK,
但团队项目需要**更强的保护**。


> **问自己**  
- 鸭子类型在什么场景下会变成'坑'?  
- 如果团队 10 个人写支付类,怎么保证每个人都写了 `execute`?  
- 能不能在**创建对象时**就检查,而不是等到调用时才报错?


> **中期小结 1** —— 到目前为止

**学到的**:  
- if-elif 类型分发 → 难以扩展  
- 鸭子类型 → 灵活但不安全  
- 问题:**如何让'必须实现的方法'强制执行**?

**下一步**:引入 abc,建立真正的契约。


#### abc.ABC 抽象基类 —— 把'必须实现'写进契约

> **类比**  
国家发布 USB-C 接口规范:每个厂商的设备必须有 type-c 口。
不符合规范 → 不准上市。

> **解释**  
- `abc.ABC` 作为父类 → 类成为**抽象基类**  
- `@abstractmethod` 装饰的方法 → 子类**必须实现**  
- 漏写 → **实例化时报 TypeError**,而不是运行时


In [ ]:
import abc

class Payment(abc.ABC):
    @abc.abstractmethod
    def execute(self, amount):
        ...

# --- 执行过程 ---
# 第 2 行 class Payment(abc.ABC):
#   ① 继承 abc.ABC → Payment 成为抽象基类
#   ② 不能直接实例化
# 第 3 行 @abc.abstractmethod:
#   ① 装饰器标记 execute 为抽象方法
#   ② 子类必须实现 execute,否则也不能实例化
# 第 7 行 Payment():
#   ① 尝试实例化抽象基类
#   ② Python 检查:有抽象方法未实现 → TypeError

try:
    p = Payment()
except TypeError as e:
    print(f"报错: {e}")


> **逐行解剖**  
① `class Payment(abc.ABC)` → 继承 abc.ABC,成为抽象基类  
② `@abc.abstractmethod` → 标记 execute 为抽象方法  
③ `Payment()` → 尝试实例化 → TypeError  
④ 错误信息明确告诉你:哪些方法没实现


> **ASCII 内存图** —— 抽象基类

```
Payment(abc.ABC)
    │
    ├── execute (抽象方法,未实现)
    │
    └── 不能直接实例化!
        │
        ▼
    TypeError: Can't instantiate abstract class
               Payment with abstract method execute

子类必须实现 execute 才能实例化
```


> **@abstractmethod** 装饰器的作用

`@abstractmethod` 是 Python 的'契约标记':  
- 标记后,该方法**只有签名,没有实现**  
- 子类**必须**提供实现  
- 实例化时 Python 会检查:所有抽象方法都实现了吗?  
- 没有 → **立即报 TypeError**


In [ ]:
import abc

class Payment(abc.ABC):
    @abc.abstractmethod
    def execute(self, amount):
        ...

class Alipay(Payment):
    def execute(self, amount):
        return f"支付宝支付 {amount} 元"

class WeChatPay(Payment):
    def execute(self, amount):
        return f"微信支付 {amount} 元"

# --- 执行过程 ---
# 第 7 行 class Alipay(Payment):
#   ① 继承 Payment → 必须实现 execute
#   ② 实现了 execute → 契约满足
# 第 13 行 Alipay():
#   ① 检查:execute 已实现 → 允许实例化
#   ② 创建成功

alipay = Alipay()
wechat = WeChatPay()
print(alipay.execute(99.0))
print(wechat.execute(50.0))


> **逐行解剖**  
① `Alipay(Payment)` 继承抽象基类 → 必须实现 execute  
② `def execute(self, amount)` → 提供实现 → 契约满足  
③ `Alipay()` → 检查通过 → 实例化成功  
④ `alipay.execute(99.0)` → 调用子类实现


> **ASCII 内存图** —— 继承链 + 方法查找

```
Payment(abc.ABC)
    │
    ├── execute (抽象)
    │
    ├── Alipay(Payment)
    │     └── execute (已实现) ← 实例化 OK
    │
    └── WeChatPay(Payment)
          └── execute (已实现) ← 实例化 OK

alipay.execute(99.0)
    │
    ▼
Alipay.execute → 找到 → 执行
```


> **常见错误**  
1. **忘记继承 abc.ABC**:见下一个 Break It  
2. **抽象方法有实现**:`@abstractmethod` 下写了代码 → 仍然可以实例化  
3. **子类漏写方法**:实例化时报 TypeError,错误信息会告诉你缺哪个方法


> **NCDL Break It #2** —— 忘记继承 abc.ABC

场景:你用了 `@abstractmethod`,但忘了继承 abc.ABC。
'能跑',但漏写 execute 不报错 → 鸭子类型的弱点又回来了!


In [ ]:
# ============ BREAK IT 演示 ============
import abc

class BrokenPayment:  # 没有 abc.ABC! BREAK IT!
    @abc.abstractmethod
    def execute(self, amount):
        ...

class Alipay(BrokenPayment):
    pass  # 漏写 execute!

alipay = Alipay()  # 竟然不报错!
print("没报错!但 execute 不存在,运行时才爆炸")
try:
    alipay.execute(99)
except AttributeError as e:
    print(f"运行时报错: {e}")
# ============ END BREAK IT ============


> **逐行解剖**  
① `class BrokenPayment:` 没有继承 abc.ABC → 不是抽象基类  
② `@abstractmethod` 装饰器**单独使用无效**  
③ `Alipay(BrokenPayment)` → 没有强制检查  
④ `Alipay()` → 实例化成功 → 看起来正常  
⑤ `alipay.execute(99)` → AttributeError → 运行时才暴露


> **关键结论**  
`@abstractmethod` + `abc.ABC` **必须一起用**!  
- 只有 `@abstractmethod` → 装饰器空操作,无强制力  
- 只有 `abc.ABC` → 没有抽象方法,类可以实例化  
- 两者结合 → 真正的契约保护


> **为什么**会这样?  
`@abstractmethod` 只是在方法上打个标记。
但检查'所有抽象方法是否都实现了'是 abc.ABC 的工作。
没有 abc.ABC,就没有检查机制 → 标记形同虚设。


> **问自己**  
- 为什么单独用 `@abstractmethod` 没有效果?  
- abc.ABC 在背后做了什么检查?  
- 如果团队有人忘了继承 abc.ABC,怎么发现?


> **中期小结 2** —— abc 契约

**学到的**:  
- abc.ABC 让类成为抽象基类  
- @abstractmethod 标记必须实现的方法  
- 两者结合 → 实例化时强制检查

**关键**:契约从'口头约定'变成'强制执行'。


#### 接口概念 —— 团队协作的契约

> **解释**  
- **接口** = 只包含抽象方法的抽象类  
- Python 用'全抽象方法的 ABC'模拟接口  
- 接口**不实现任何方法**,只定义'必须做什么'  
- 子类**实现接口**,提供具体行为

> **类比**  
接口 = 合同条款,实现类 = 签约方。
合同只写'乙方必须做什么',不管'怎么做'。


In [ ]:
import abc

class Notifier(abc.ABC):
    @abc.abstractmethod
    def send(self, msg):
        ...

    @abc.abstractmethod
    def channel(self):
        ...

# --- 执行过程 ---
# 第 2 行 class Notifier(abc.ABC):
#   ① 继承 abc.ABC → 抽象基类
#   ② 所有方法都是抽象的 → 这就是接口
# 第 3-4 行 @abc.abstractmethod:
#   ① send 和 channel 都是抽象方法
#   ② 子类必须实现这两个方法

try:
    n = Notifier()
except TypeError as e:
    print(f"接口不能实例化: {e}")


> **逐行解剖**  
① `Notifier(abc.ABC)` → 抽象基类  
② 所有方法都标记 `@abstractmethod` → 这就是**接口**  
③ `Notifier()` → 不能实例化 → 错误信息列出所有抽象方法  
④ 子类必须实现 `send` 和 `channel` 才能实例化


> **ASCII 内存图** —— 接口定义

```
Notifier(abc.ABC)  ← 接口(全部抽象)
    │
    ├── send (抽象) ── 子类必须实现
    └── channel (抽象) ── 子类必须实现

不能实例化!只能被实现。
```


In [ ]:
import abc

class Notifier(abc.ABC):
    @abc.abstractmethod
    def send(self, msg):
        ...

    @abc.abstractmethod
    def channel(self):
        ...

class EmailNotifier(Notifier):
    def send(self, msg):
        return f"邮件: {msg}"

    def channel(self):
        return "email"

class SMSNotifier(Notifier):
    def send(self, msg):
        return f"短信: {msg}"

    def channel(self):
        return "sms"

# --- 执行过程 ---
# 第 12 行 EmailNotifier(Notifier):
#   ① 继承 Notifier → 必须实现 send 和 channel
#   ② 两个方法都实现了 → 契约满足
# 第 24 行 for n in [...]:
#   ① 遍历两个实例
#   ② n.channel() → 多态调用各自的实现
#   ③ n.send(...) → 多态调用各自的实现

for n in [EmailNotifier(), SMSNotifier()]:
    print(f"[{n.channel()}] {n.send('订单已发货')}")


> **逐行解剖**  
① `EmailNotifier` 实现 `send` 和 `channel` → 契约满足  
② `SMSNotifier` 同样实现 → 契约满足  
③ `for n in [...]` → 多态:同一调用,不同行为  
④ `n.channel()` → 各自返回自己的渠道名


> **ASCII 内存图** —— 接口继承链

```
Notifier(abc.ABC)  ← 接口
    │
    ├── send (抽象)
    ├── channel (抽象)
    │
    ├── EmailNotifier
    │     ├── send → "邮件: ..."
    │     └── channel → "email"
    │
    └── SMSNotifier
          ├── send → "短信: ..."
          └── channel → "sms"

for n in [EmailNotifier(), SMSNotifier()]:
    n.send(msg)  ← 多态:同一调用,不同实现
```


> **练习** —— Serializer 接口

定义接口 `Serializer`,抽象方法 `serialize(obj)` 和 `deserialize(text)`。
子类 `JsonSerializer` 简单模拟格式。

> 问自己:  
- 接口应该继承什么?  
- 抽象方法用什么装饰器?  
- 子类要实现哪些方法才能实例化?


In [ ]:
# ============ 学员代码区 ============
import abc

class Serializer(abc.ABC):
    # 请定义 serialize 和 deserialize 抽象方法
    pass

class JsonSerializer(Serializer):
    # 请实现 serialize 和 deserialize
    pass

# s = JsonSerializer()
# print(s.serialize("hello"))
pass

# ============ 参考答案 ============
import abc

class Serializer(abc.ABC):
    @abc.abstractmethod
    def serialize(self, obj):
        ...

    @abc.abstractmethod
    def deserialize(self, text):
        ...

class JsonSerializer(Serializer):
    def serialize(self, obj):
        return "{data: " + str(obj) + "}"

    def deserialize(self, text):
        return text

s = JsonSerializer()
print(s.serialize("hello"))


> **逐行解剖**  
① `Serializer(abc.ABC)` → 接口基类  
② `@abc.abstractmethod` → 标记两个抽象方法  
③ `JsonSerializer(Serializer)` → 实现两个方法  
④ `s.serialize("hello")` → 调用具体实现


#### 鸭子类型 vs abc.ABC —— 什么时候用哪个?

> **取舍原则**  
- **个人脚本/小项目** → 鸭子类型,灵活快速  
- **团队协作/大项目** → abc.ABC,强制契约  
- **公共库/API** → abc.ABC,明确接口

> **类比**  
鸭子类型 = 口头约定('你应该有这个方法')  
abc.ABC = 合同('你必须实现这个方法,否则不准上线')


In [ ]:
# 鸭子类型版本 —— 灵活但不安全
class Alipay:
    def execute(self, amount):
        return f"支付宝 {amount} 元"

def checkout(total, payment):
    return payment.execute(total)

print(checkout(99.0, Alipay()))
# 优点:简单,无需 import abc
# 缺点:漏写 execute 不报错


In [ ]:
# abc.ABC 版本 —— 安全但稍繁琐
import abc

class Payment(abc.ABC):
    @abc.abstractmethod
    def execute(self, amount):
        ...

class Alipay(Payment):
    def execute(self, amount):
        return f"支付宝 {amount} 元"

def checkout(total, payment):
    return payment.execute(total)

print(checkout(99.0, Alipay()))
# 优点:漏写 execute 实例化时报错
# 缺点:需要 import abc,多写几行


> **对比总结**

| 维度 | 鸭子类型 | abc.ABC |
| --- | --- | --- |
| 灵活性 | 高 | 中 |
| 安全性 | 低(运行时才报错) | 高(实例化时报错) |
| 代码量 | 少 | 稍多 |
| 团队协作 | 容易出错 | 强制契约 |
| 适用场景 | 个人脚本 | 团队项目 |

> **经验法则**  
超过 3 个人的项目,用 abc.ABC。
个人脚本/原型,用鸭子类型。


> **NCDL Break It #3** —— 子类漏实现抽象方法

场景:子类继承抽象基类,但漏实现了某个抽象方法。
Python 会在**实例化时**告诉你缺了什么。


In [ ]:
# ============ BREAK IT 演示 ============
import abc

class Payment(abc.ABC):
    @abc.abstractmethod
    def execute(self, amount):
        ...

    @abc.abstractmethod
    def refund(self, order_id):
        ...

class Alipay(Payment):
    def execute(self, amount):
        return f"支付宝支付 {amount} 元"
    # 注意:漏写了 refund!

try:
    alipay = Alipay()
except TypeError as e:
    print(f"报错: {e}")
    print("错误信息明确告诉你:缺了 refund 方法")
# ============ END BREAK IT ============


> **逐行解剖**  
① `Payment` 有两个抽象方法:`execute` 和 `refund`  
② `Alipay` 只实现了 `execute`,漏了 `refund`  
③ `Alipay()` → 实例化时检查 → 发现 `refund` 未实现  
④ TypeError → 错误信息明确列出缺失的方法  
⑤ **好处**:问题在**创建对象时**就暴露,不会藏到运行时


> **常见错误**  
1. **漏写方法**:子类只实现部分抽象方法 → 实例化时报错  
2. **方法签名不一致**:参数不同 → Python 不检查签名,但调用时会出错  
3. **忘记 import abc**:`NameError: name 'abc' is not defined`


> **问自己**  
- 为什么 Python 在实例化时检查,而不是在类定义时?  
- 如果抽象方法有 5 个,子类只实现了 3 个,错误信息会怎么说?  
- 这种'提前报错'比'运行时才报错'好在哪里?


> **NCDL Break It #4** —— 尝试实例化抽象基类

直接 `Payment()` 会怎样?让我们实测。


In [ ]:
# ============ BREAK IT 演示 ============
import abc

class Payment(abc.ABC):
    @abc.abstractmethod
    def execute(self, amount):
        ...

try:
    p = Payment()
except TypeError as e:
    print(f"直接实例化抽象基类 → {e}")
# ============ END BREAK IT ============


> **逐行解剖**  
① `Payment(abc.ABC)` → 抽象基类,有抽象方法未实现  
② `Payment()` → 尝试实例化 → TypeError  
③ 错误信息:`Can't instantiate abstract class Payment
   with abstract method execute`  
④ **结论**:抽象基类是'图纸',不能直接用,必须通过子类


> **中期小结 3** —— 三种错误时机对比

| 错误类型 | 什么时候报错 | 鸭子类型 | abc.ABC |
| --- | --- | --- | --- |
| 漏写方法 | 实例化时/运行时 | 运行时 | 实例化时 |
| 拼写错误 | 运行时 | 运行时 | 运行时 |
| 直接实例化 | 实例化时 | 能实例化 | 报错 |

**abc.ABC 的优势**:让'漏写方法'在**更早**的阶段报错。


#### 综合练习:Payment 支付系统 (L3 完整落地)

**目标**:把 Day09 末尾那个有 if-elif 味的系统,重构为多态版本。

**要求**:  
- `Payment(abc.ABC)` 定义契约  
- `checkout(cart_total, payment)` **不使用 if/elif/isinstance/type**  
- 漏写 `execute` 时在实例化阶段就报 TypeError  
- 新增支付方式只需**添加一个文件**


> **问自己**  
- 这道题要用到今天学的哪个知识点?  
- 抽象基类应该定义哪些抽象方法?  
- `checkout` 函数应该怎么写才不包含 if-elif?  
- 如果运行报错,检查:继承 abc.ABC 了吗? 实现了所有抽象方法吗?


In [ ]:
# ============ 学员代码区 ============
import abc

class Payment(abc.ABC):
    # 请定义 execute 抽象方法
    pass

class Alipay(Payment):
    # 请实现 execute 方法
    pass

class WeChatPay(Payment):
    # 请实现 execute 方法
    pass

def checkout(cart_total, payment):
    # 请实现 checkout(不判断类型)
    pass

# 测试
# print(checkout(99.0, Alipay()))
# print(checkout(50.0, WeChatPay()))
pass


> **提示**  
1. 抽象基类 `Payment` 继承 `abc.ABC`  
2. 用 `@abc.abstractmethod` 标记 `execute`  
3. `Alipay` 和 `WeChatPay` 继承 `Payment` 并实现 `execute`  
4. `checkout` 直接调用 `payment.execute(cart_total)`


In [ ]:
# ============ 参考答案 ============
import abc

class Payment(abc.ABC):
    @abc.abstractmethod
    def execute(self, amount):
        ...

class Alipay(Payment):
    def execute(self, amount):
        return f"支付宝支付 {amount:.2f} 元"

class WeChatPay(Payment):
    def execute(self, amount):
        return f"微信支付 {amount:.2f} 元"

def checkout(cart_total, payment):
    return payment.execute(cart_total)

# --- 执行过程 ---
# 第 18 行 checkout(99.0, Alipay()):
#   ① 创建 Alipay 实例
#   ② 调用 checkout → payment 指向该实例
#   ③ payment.execute(99.0) → Alipay.execute
#   ④ 返回 "支付宝支付 99.00 元"
# 第 19 行 checkout(50.0, WeChatPay()):同理

print(checkout(99.0, Alipay()))
print(checkout(50.0, WeChatPay()))

# 验证:漏写 execute 会报错
class BrokenPay(Payment):
    pass

try:
    broken = BrokenPay()
except TypeError as e:
    print(f"契约保护: {e}")


> **逐行解剖**  
① `Payment(abc.ABC)` → 抽象基类,定义契约  
② `@abc.abstractmethod` → execute 是必须实现的方法  
③ `Alipay(Payment)` → 继承并实现 execute → 契约满足  
④ `checkout` → 不判断类型,直接调用 execute  
⑤ `BrokenPay(Payment)` → 没实现 execute → 实例化时报错


> **ASCII 内存图** —— 完整支付系统

```
Payment(abc.ABC)  ← 契约
    │
    ├── execute (抽象)
    │
    ├── Alipay
    │     └── execute → "支付宝支付 ..."
    │
    └── WeChatPay
          └── execute → "微信支付 ..."

checkout(99.0, Alipay())
    │
    ▼
payment.execute(99.0)  ← 多态调用
    │
    ▼
Alipay.execute → 返回结果

新增支付方式?只需加一个类,实现 execute 即可
checkout 函数零修改!
```


> **扩展思考** —— 新增支付方式

多态的好处:新增一种支付,**原有代码零修改**。


In [ ]:
# 新增一种支付方式 —— 只需加一个类!
class ApplePay(Payment):
    def execute(self, amount):
        return f"Apple Pay 支付 {amount:.2f} 元"

print(checkout(120.0, ApplePay()))
# checkout 函数完全没有修改!
# 这就是'开闭原则':对扩展开放,对修改关闭


#### 今日小结

| 概念 | 解决的痛点 | 适用场景 |
| --- | --- | --- |
| 鸭子类型 | 不写继承也能多态 | 个人脚本、原型 |
| abc.ABC 抽象基类 | 漏实现立即报错 | 团队项目、公共库 |
| @abstractmethod | 标记必须实现的方法 | 配合 abc.ABC 使用 |
| 接口 | 团队协作契约 | 多人协作、API 设计 |
| NCDL 负案例 | 通过踩坑加深理解 | 所有抽象概念 |

> **核心洞察**  
多态 = 同一接口,不同实现。
契约 = 强制子类实现接口。
鸭子类型灵活但不安全,abc.ABC 安全但稍繁琐。
**根据项目规模选择**。


> **学习检查清单**  
完成今天的学习后,你应该能:  
- [ ] 解释鸭子类型的工作原理  
- [ ] 说出鸭子类型的优点和缺点  
- [ ] 用 abc.ABC 定义抽象基类  
- [ ] 用 @abstractmethod 标记必须实现的方法  
- [ ] 解释为什么 abc.ABC 和 @abstractmethod 必须一起用  
- [ ] 说出什么时候用鸭子类型,什么时候用 abc.ABC  
- [ ] 解释接口的概念和作用  
- [ ] 独立实现一个基于 abc.ABC 的多态系统


> **更多练习**  
- 当堂练:course/lesson10/in_class/practice01-06.py  
- 课后作业:course/lesson10/homework/task01-03.py
